# XOM / XLE — modello passo per passo

| # | Contenuto |
|---|-----------|
| 1 | Download XOM/XLE (yfinance) → CSV locale |
| 2 | OLS rolling su 60 rendimenti precedenti: β e residui ε (funzione + test) |
| 3 | Cumulata dei residui, AR(1) sulle 59 coppie → κ, m, σ_eq, filtro b < 0.9672 |
| 4 | s-score e grafico con le soglie |
| 5 | s vs −media(X)/std(X): scatter, correlazione, conclusione |
| 6 | Simulazione OU con κ noto, 1000 stime, JSON di riferimento |

In [1]:
from __future__ import annotations

import json
import logging
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from scipy import stats

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("xom_xle")

# ---- Parametri del modello (dalla scheda del task) -------------------------
STOCK, ETF = "XOM", "XLE"
AS_OF = date(2026, 11, 23)       # ultimo giorno di borsa completo (D-1): nessun "oggi" implicito
YEARS = 3
WINDOW = 60                      # rendimenti per finestra -> 60 residui -> X_1..X_60 -> 59 coppie AR(1)
TRADING_DAYS = 252               # dt = 1/252
B_MAX = 0.9672                   # filtro b < 0.9672  <=>  kappa > ~8.4  (mean reversion < ~30 giorni)
S_OPEN_SHORT, S_OPEN_LONG = 1.25, -1.25
S_CLOSE_LONG, S_CLOSE_SHORT = -0.50, 0.75

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PRICES_CSV = DATA_DIR / "xom_xle_daily.csv"
OU_REFERENCE_JSON = DATA_DIR / "ou_reference.json"

assert abs(-TRADING_DAYS * np.log(B_MAX) - 8.4) < 0.01, "B_MAX deve corrispondere a kappa ~ 8.4"

## 1 · Dati: XOM e XLE, ~3 anni daily

Il download parte solo se il CSV locale non esiste; altrimenti si rilegge il file (nessuna chiamata di rete).
Prezzi *adjusted* (`auto_adjust=True`).

In [2]:
class DataFetchError(RuntimeError):
    '''Raised when yfinance returns no usable data.'''


def download_prices(tickers: list[str], as_of: date, years: int = YEARS) -> pd.DataFrame:
    '''Download daily adjusted close prices from yfinance.

    Parameters
    ----------
    tickers : list[str]
        Tickers to download.
    as_of : date
        Last calendar day (inclusive) to include.
    years : int
        Approximate history length in years.

    Returns
    -------
    pd.DataFrame
        Adjusted close prices, one column per ticker, DatetimeIndex named ``date``.

    Raises
    ------
    DataFetchError
        If yfinance returns nothing or a ticker is missing.

    Notes
    -----
    ``end`` is exclusive in yfinance, hence ``as_of + 1 day``.
    '''
    import yfinance as yf  # import locale: serve solo al primo download

    start = as_of - timedelta(days=365 * years)
    raw = yf.download(
        tickers,
        start=start.isoformat(),
        end=(as_of + timedelta(days=1)).isoformat(),
        auto_adjust=True,
        progress=False,
    )
    if raw.empty:
        raise DataFetchError(f"yfinance ha restituito un DataFrame vuoto per {tickers}")
    prices = raw["Close"][tickers].dropna()
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    prices.index.name = "date"
    return prices


if PRICES_CSV.exists():
    prices = pd.read_csv(PRICES_CSV, index_col="date", parse_dates=True)
    logger.info("Prezzi letti da %s", PRICES_CSV)
else:
    prices = download_prices([STOCK, ETF], AS_OF)
    prices.to_csv(PRICES_CSV)
    logger.info("Prezzi scaricati e salvati in %s", PRICES_CSV)

prices = prices.loc[: pd.Timestamp(AS_OF), [STOCK, ETF]]

assert prices.index.is_monotonic_increasing and prices.index.is_unique
assert prices.notna().all().all(), "NaN nei prezzi"
assert (prices > 0).all().all(), "prezzi non positivi"
assert len(prices) >= 3 * 200, f"troppo poche osservazioni: {len(prices)}"
assert prices.index.max() <= pd.Timestamp(AS_OF)

returns = prices.pct_change().dropna()
logger.info("%d prezzi (%s -> %s), %d rendimenti semplici",
            len(prices), prices.index[0].date(), prices.index[-1].date(), len(returns))
prices.tail()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/utils.py:507: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/utils.py:521: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Lib

Ticker,XOM,XLE
date,,
2026-11-17,167.210007,63.360001
2026-11-18,160.580002,61.290001
2026-11-19,158.929993,60.889999
2026-11-20,159.990005,60.509998
2026-11-23,156.899994,59.869999


## 2 · OLS rolling: β e residui ε

Per ogni giorno *t*: regressione OLS **con intercetta** dei 60 rendimenti di XOM sui 60 rendimenti di XLE dei giorni
*t−60 … t−1* (il rendimento del giorno *t* non entra: niente lookahead). Si salvano α, β e i 60 residui.

Con l'intercetta i residui hanno somma esattamente zero, quindi $X_{60}=0$ (serve al punto 4).

In [3]:
class WindowError(ValueError):
    '''Raised when a regression window is degenerate or malformed.'''


@dataclass(frozen=True)
class OLSFit:
    '''Result of one OLS fit ``stock = alpha + beta * etf + eps``.'''

    alpha: float
    beta: float
    residuals: np.ndarray


def fit_ols(stock: np.ndarray, etf: np.ndarray) -> OLSFit:
    '''OLS of a stock's returns on an ETF's returns, with intercept.

    Parameters
    ----------
    stock : np.ndarray
        1-D array of stock returns (dependent variable).
    etf : np.ndarray
        1-D array of ETF returns (regressor), same length as ``stock``.

    Returns
    -------
    OLSFit
        Intercept, slope and residuals ``stock - alpha - beta * etf``.

    Raises
    ------
    WindowError
        If the inputs are not 1-D arrays of equal length >= 3, or the ETF is constant.

    Notes
    -----
    Closed form ``beta = cov(x, y) / var(x)``. With an intercept the residuals sum to
    zero by construction, so the cumulated residual ``X`` ends at 0 on the last day.
    '''
    if stock.ndim != 1 or stock.shape != etf.shape or len(stock) < 3:
        raise WindowError(f"finestra non valida: {stock.shape} vs {etf.shape}")
    xc = etf - etf.mean()
    var = float(xc @ xc)
    if var == 0.0:
        raise WindowError("ETF costante nella finestra: beta non identificato")
    beta = float(xc @ (stock - stock.mean()) / var)
    alpha = float(stock.mean() - beta * etf.mean())
    return OLSFit(alpha=alpha, beta=beta, residuals=stock - alpha - beta * etf)


@dataclass(frozen=True)
class RollingFits:
    '''Rolling OLS output. Every row is a day t, built only from returns before t.'''

    alphas: pd.Series
    betas: pd.Series
    residuals: pd.DataFrame  # shape (n_days, window); column j = residual of return t-window+j


def rolling_ols(
    returns: pd.DataFrame, stock: str, etf: str, as_of: date, window: int = WINDOW
) -> RollingFits:
    '''Rolling OLS of ``stock`` on ``etf`` over the ``window`` previous returns.

    Parameters
    ----------
    returns : pd.DataFrame
        Daily returns, DatetimeIndex, with columns ``stock`` and ``etf``.
    stock, etf : str
        Column names.
    as_of : date
        Only returns dated <= ``as_of`` are used (applied first).
    window : int
        Number of previous returns per regression.

    Returns
    -------
    RollingFits
        One row per day t >= ``window`` (position), indexed by t.

    Raises
    ------
    WindowError
        If there are not more than ``window`` returns before ``as_of``.

    Notes
    -----
    The regression at day t uses positions ``t-window .. t-1``: the return of day t is
    never included.
    '''
    returns = returns.loc[returns.index <= pd.Timestamp(as_of)]
    assert returns.index.max() <= pd.Timestamp(as_of)
    if len(returns) <= window:
        raise WindowError(f"servono piu' di {window} rendimenti, trovati {len(returns)}")
    y, x = returns[stock].to_numpy(), returns[etf].to_numpy()
    dates = returns.index[window:]
    alphas = np.empty(len(dates))
    betas = np.empty(len(dates))
    resid = np.empty((len(dates), window))
    for k, i in enumerate(range(window, len(returns))):
        fit = fit_ols(y[i - window : i], x[i - window : i])
        alphas[k], betas[k], resid[k] = fit.alpha, fit.beta, fit.residuals
    return RollingFits(
        alphas=pd.Series(alphas, index=dates, name="alpha"),
        betas=pd.Series(betas, index=dates, name="beta"),
        residuals=pd.DataFrame(resid, index=dates),
    )


fits = rolling_ols(returns, STOCK, ETF, AS_OF)
logger.info("%d finestre; beta: media %.3f, min %.3f, max %.3f",
            len(fits.betas), fits.betas.mean(), fits.betas.min(), fits.betas.max())
fits.betas.describe().to_frame().T

INFO xom_xle: 691 finestre; beta: media 0.981, min 0.735, max 1.216


,count,mean,std,min,25%,50%,75%,max
beta,691.0,0.980926,0.107198,0.735477,0.928809,0.984909,1.040813,1.215536


**Test del punto 2** (asserzioni end-to-end, come da linee guida):

In [4]:
rng_t = np.random.default_rng(0)

# (a) relazione esatta senza rumore: recupero di alpha, beta e residui nulli
x_t = rng_t.normal(0, 0.01, WINDOW)
fit_t = fit_ols(0.0003 + 1.7 * x_t, x_t)
assert np.isclose(fit_t.beta, 1.7, atol=1e-12) and np.isclose(fit_t.alpha, 0.0003, atol=1e-12)
assert np.abs(fit_t.residuals).max() < 1e-12

# (b) identico a un least-squares indipendente (np.linalg.lstsq) su una finestra reale (ultima)
y_w = returns[STOCK].to_numpy()[-WINDOW - 1 : -1]
x_w = returns[ETF].to_numpy()[-WINDOW - 1 : -1]
design = np.column_stack([np.ones(WINDOW), x_w])
coef, *_ = np.linalg.lstsq(design, y_w, rcond=None)
mine = fit_ols(y_w, x_w)
assert np.allclose([mine.alpha, mine.beta], coef, atol=1e-12)
assert np.allclose(mine.residuals, y_w - design @ coef, atol=1e-12)
# la finestra dell'ultimo giorno e' proprio questa
assert np.isclose(fits.betas.iloc[-1], mine.beta, atol=1e-12)

# (c) proprieta' dei residui OLS con intercetta: somma zero e ortogonali al regressore
assert np.abs(fits.residuals.sum(axis=1)).max() < 1e-10
assert abs(float(fits.residuals.iloc[-1].to_numpy() @ x_w)) < 1e-10

# (d) forma e allineamento: prima data = (WINDOW+1)-esimo rendimento
assert fits.residuals.shape == (len(returns) - WINDOW, WINDOW)
assert fits.betas.index[0] == returns.index[WINDOW]

# (e) nessun lookahead: perturbare il rendimento del giorno t non cambia il fit di t,
#     ma cambia quello di t+1
i = 100
bumped = returns.copy()
bumped.iloc[i, bumped.columns.get_loc(STOCK)] += 0.05
fits_b = rolling_ols(bumped, STOCK, ETF, AS_OF)
d_t, d_t1 = returns.index[i], returns.index[i + 1]
assert fits_b.betas[d_t] == fits.betas[d_t], "lookahead: il fit del giorno t dipende dal rendimento di t"
assert fits_b.betas[d_t1] != fits.betas[d_t1]

# (f) input degenere -> eccezione di dominio
try:
    fit_ols(np.arange(5.0), np.ones(5))
except WindowError:
    pass
else:
    raise AssertionError("WindowError attesa per ETF costante")

logger.info("Punto 2: tutti i test superati")

INFO xom_xle: Punto 2: tutti i test superati


## 3 · Cumulata dei residui e parametri OU

Per ogni finestra: $X_k=\sum_{j\le k}\varepsilon_j$ ($k=1\ldots60$), regressione AR(1) sulle 59 coppie
$X_{n+1}=a+b\,X_n+\zeta_n$, poi

$$\kappa=-252\ln b,\qquad m=\frac{a}{1-b},\qquad \sigma_{eq}=\frac{\sigma_\zeta}{\sqrt{1-b^2}}$$

* $\sigma_\zeta$ = deviazione standard dei residui dell'AR(1) con **ddof = 2** (due parametri stimati, n−2 = 57 gradi di libertà).
* Filtro: tengo la finestra solo se $0<b<0.9672$ ($\kappa>8.4$). Per $b\le0$ il logaritmo non esiste e per
  $b\ge1$ il processo non è mean-reverting: in entrambi i casi κ, m, σ_eq sono `NaN` e la finestra è scartata.

In [5]:
@dataclass(frozen=True)
class OUParams:
    '''Daily OU parameters estimated from one AR(1) fit.'''

    a: float
    b: float
    kappa: float
    m: float
    sigma_eq: float
    valid: bool


def estimate_ou_from_x(x: np.ndarray) -> OUParams:
    '''Estimate OU parameters from a cumulated-residual series via AR(1).

    Parameters
    ----------
    x : np.ndarray
        1-D series ``X_1..X_n`` (n = 60 -> 59 pairs).

    Returns
    -------
    OUParams
        ``a, b`` always; ``kappa, m, sigma_eq`` are NaN unless ``0 < b < 1``.
        ``valid`` is True iff ``0 < b < B_MAX``.

    Raises
    ------
    WindowError
        If fewer than 5 points are given or ``X[:-1]`` is constant.

    Notes
    -----
    ``sigma_zeta`` uses ddof=2 (OLS with 2 parameters). ``kappa = -252 ln b``,
    ``m = a / (1 - b)``, ``sigma_eq = sigma_zeta / sqrt(1 - b^2)``.
    On short windows ``b`` is biased downward, so ``kappa`` is biased upward (see step 6).
    '''
    if x.ndim != 1 or len(x) < 5:
        raise WindowError(f"serie troppo corta o non 1-D: {x.shape}")
    x0, x1 = x[:-1], x[1:]
    n = len(x0)
    x0c = x0 - x0.mean()
    var = float(x0c @ x0c)
    if var == 0.0:
        raise WindowError("X costante: AR(1) non identificato")
    b = float(x0c @ (x1 - x1.mean()) / var)
    a = float(x1.mean() - b * x0.mean())
    zeta = x1 - a - b * x0
    sigma_zeta = float(np.sqrt(zeta @ zeta / (n - 2)))
    if 0.0 < b < 1.0:
        kappa = float(-TRADING_DAYS * np.log(b))
        m = a / (1.0 - b)
        sigma_eq = sigma_zeta / float(np.sqrt(1.0 - b * b))
    else:
        kappa = m = sigma_eq = float("nan")
    return OUParams(a=a, b=b, kappa=kappa, m=m, sigma_eq=sigma_eq, valid=bool(0.0 < b < B_MAX))


def estimate_ou(residuals: np.ndarray) -> OUParams:
    '''Cumulate regression residuals into X and estimate the OU parameters.

    Parameters
    ----------
    residuals : np.ndarray
        1-D regression residuals of one window.

    Returns
    -------
    OUParams
        See :func:`estimate_ou_from_x`.

    Raises
    ------
    WindowError
        Propagated from :func:`estimate_ou_from_x`.
    '''
    return estimate_ou_from_x(np.cumsum(residuals))


rows = [estimate_ou(r) for r in fits.residuals.to_numpy()]
ou = pd.DataFrame([vars(p) for p in rows], index=fits.residuals.index)
ou.insert(0, "beta", fits.betas)
logger.info("Finestre valide (0 < b < %.4f): %d / %d (%.1f%%)",
            B_MAX, ou["valid"].sum(), len(ou), 100 * ou["valid"].mean())
logger.info("Scartate: b<=0: %d | b>=1: %d | 0.9672<=b<1: %d",
            (ou["b"] <= 0).sum(), (ou["b"] >= 1).sum(), ((ou["b"] >= B_MAX) & (ou["b"] < 1)).sum())
ou[ou["valid"]].describe().T[["count", "mean", "std", "min", "50%", "max"]]

INFO xom_xle: Finestre valide (0 < b < 0.9672): 691 / 691 (100.0%)
INFO xom_xle: Scartate: b<=0: 0 | b>=1: 0 | 0.9672<=b<1: 0


,count,mean,std,min,50%,max
beta,691.0,0.980926,0.107198,0.735477,0.984909,1.215536
a,691.0,-0.000112,0.001628,-0.007617,-0.000117,0.004355
b,691.0,0.830878,0.099180,0.317144,0.855871,0.954354
kappa,691.0,48.924753,35.734857,11.773492,39.220315,289.396400
m,691.0,-0.000661,0.011353,-0.030472,-0.000764,0.036084
sigma_eq,691.0,0.012113,0.003812,0.004315,0.011897,0.023773


**Test del punto 3:**

In [6]:
# (a) AR(1) deterministico (senza rumore): a e b recuperati esattamente
def ar1_path(a: float, b: float, x0: float, n: int) -> np.ndarray:
    out = np.empty(n)
    out[0] = x0
    for t in range(n - 1):
        out[t + 1] = a + b * out[t]
    return out

p = estimate_ou_from_x(ar1_path(0.01, 0.9, 1.0, WINDOW))
assert np.isclose(p.a, 0.01, atol=1e-9) and np.isclose(p.b, 0.9, atol=1e-9)
assert np.isclose(p.kappa, -TRADING_DAYS * np.log(0.9), atol=1e-6)
assert np.isclose(p.m, 0.01 / (1 - 0.9), atol=1e-6) and p.sigma_eq < 1e-9 and p.valid

# (b) filtro: b=0.99 scartato, b=0.5 tenuto, b<0 (oscillante) scartato con kappa NaN
assert not estimate_ou_from_x(ar1_path(0.0, 0.99, 1.0, WINDOW)).valid
assert estimate_ou_from_x(ar1_path(0.0, 0.5, 1.0, WINDOW)).valid
neg = estimate_ou_from_x(ar1_path(0.0, -0.5, 1.0, WINDOW))
assert not neg.valid and np.isnan(neg.kappa)

# (c) identico a lstsq sull'ultima finestra reale (a, b, sigma_zeta con n-2 gdl)
x_last = np.cumsum(fits.residuals.iloc[-1].to_numpy())
d_ar = np.column_stack([np.ones(len(x_last) - 1), x_last[:-1]])
coef_ar, *_ = np.linalg.lstsq(d_ar, x_last[1:], rcond=None)
ssr = float(np.sum((x_last[1:] - d_ar @ coef_ar) ** 2))
df_resid = len(x_last) - 1 - 2
p_last = estimate_ou_from_x(x_last)
assert len(x_last) == WINDOW and len(x_last) - 1 == 59 and df_resid == 57
assert np.allclose([p_last.a, p_last.b], coef_ar, atol=1e-12)
if 0 < p_last.b < 1:
    assert np.isclose(p_last.sigma_eq, np.sqrt(ssr / df_resid) / np.sqrt(1 - p_last.b**2), rtol=1e-10)
    assert np.isclose(p_last.m, coef_ar[0] / (1 - coef_ar[1]), rtol=1e-10)

# (d) X_60 = 0 su tutte le finestre (OLS con intercetta)
X_all = fits.residuals.cumsum(axis=1).to_numpy()
assert np.abs(X_all[:, -1]).max() < 1e-10

# (e) coerenza: valid <=> 0<b<B_MAX <=> kappa > 8.4 (con kappa definito)
v = ou[ou["valid"]]
assert (v["b"] > 0).all() and (v["b"] < B_MAX).all() and (v["kappa"] > 8.4 - 0.01).all()
assert v[["kappa", "m", "sigma_eq"]].notna().all().all() and (v["sigma_eq"] > 0).all()
assert ou["valid"].any(), "nessuna finestra valida: controllare i dati"

logger.info("Punto 3: tutti i test superati")

INFO xom_xle: Punto 3: tutti i test superati


## 4 · s-score

Poiché $X_{60}=0$:

$$s=\frac{X_{60}-m}{\sigma_{eq}}=-\frac{m}{\sigma_{eq}}$$

Soglie: **apro long** se $s<-1.25$, **apro short** se $s>+1.25$, **chiudo il long** se $s>-0.50$,
**chiudo lo short** se $s<+0.75$. Solo le finestre valide (filtro su b) hanno un s-score.

In [7]:
ou["s"] = np.where(ou["valid"], -ou["m"] / ou["sigma_eq"], np.nan)

# controllo: la formula generale (X_60 - m)/sigma_eq coincide con -m/sigma_eq
gen = (X_all[:, -1] - ou["m"].to_numpy()) / ou["sigma_eq"].to_numpy()
assert np.allclose(gen[ou["valid"].to_numpy()], ou.loc[ou["valid"], "s"].to_numpy(), atol=1e-9)
assert ou["s"].notna().sum() == ou["valid"].sum()


def plot_s_score(s: pd.Series) -> go.Figure:
    '''Plot the s-score series with the four trading thresholds.

    Parameters
    ----------
    s : pd.Series
        s-score indexed by date (NaN where the OU filter rejected the window).

    Returns
    -------
    go.Figure
        Plotly figure.
    '''
    fig = go.Figure(go.Scatter(x=s.index, y=s, mode="lines+markers", name="s",
                               marker={"size": 3}, line={"width": 1}))
    levels = [
        (S_OPEN_SHORT, "apri short  +1.25", "crimson", "dash"),
        (S_CLOSE_SHORT, "chiudi short  +0.75", "crimson", "dot"),
        (S_CLOSE_LONG, "chiudi long  −0.50", "seagreen", "dot"),
        (S_OPEN_LONG, "apri long  −1.25", "seagreen", "dash"),
    ]
    for y, label, color, dash in levels:
        fig.add_hline(y=y, line={"color": color, "dash": dash, "width": 1.2},
                      annotation_text=label, annotation_position="top left")
    fig.update_layout(title=f"s-score {STOCK} vs {ETF} (finestra {WINDOW} giorni, solo finestre con b < {B_MAX})",
                      xaxis_title="data", yaxis_title="s", template="plotly_white", height=480)
    return fig


plot_s_score(ou["s"]).show()
ou["s"].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
s,691.0,0.059334,0.893785,-2.421179,-0.582268,0.085226,0.72993,2.298553


## 5 · L'approssimazione $s\approx -\overline{X}/\mathrm{std}(X)$ regge?

Alternativa senza AR(1): lo z-score di $X_{60}=0$ rispetto alla distribuzione campionaria di X nella finestra,
$(0-\overline{X})/\mathrm{std}(X)$. Coincide con *s* se $m\approx\overline X$ e $\sigma_{eq}\approx\mathrm{std}(X)$,
cioè se la media e la dispersione stimate dall'AR(1) sono vicine a quelle campionarie. Può non essere vero
perché $m=a/(1-b)$ è instabile quando $b$ è vicino a 1, e $b$ è distorto sulle finestre corte.

Confronto sulle stesse finestre valide. Criterio (mio, da rivedere): **regge** se Pearson ≥ 0.95 *e* le due misure
mettono il giorno nella stessa zona di trading (cinque zone delimitate dalle soglie del punto 4) in ≥ 85% dei casi.

In [8]:
x_mean = X_all.mean(axis=1)
x_std = X_all.std(axis=1, ddof=1)
ou["s_approx"] = -x_mean / x_std

cmp = ou.loc[ou["valid"], ["s", "s_approx"]]
r_p = float(stats.pearsonr(cmp["s"], cmp["s_approx"]).statistic)
r_s = float(stats.spearmanr(cmp["s"], cmp["s_approx"]).statistic)
slope, intercept = np.polyfit(cmp["s"], cmp["s_approx"], 1)
mae = float((cmp["s"] - cmp["s_approx"]).abs().mean())

edges = [S_OPEN_LONG, S_CLOSE_LONG, S_CLOSE_SHORT, S_OPEN_SHORT]
zone_agreement = float((np.digitize(cmp["s"], edges) == np.digitize(cmp["s_approx"], edges)).mean())

assert np.isfinite([r_p, r_s, slope, mae]).all() and 0.0 <= zone_agreement <= 1.0
assert len(cmp) == int(ou["valid"].sum())

lim = float(np.nanmax(np.abs(cmp.to_numpy()))) * 1.05
fig = go.Figure()
fig.add_trace(go.Scatter(x=cmp["s"], y=cmp["s_approx"], mode="markers", name="giorni",
                         marker={"size": 5, "opacity": 0.55}, text=cmp.index.strftime("%Y-%m-%d")))
fig.add_trace(go.Scatter(x=[-lim, lim], y=[-lim, lim], mode="lines", name="y = x",
                         line={"dash": "dash", "color": "gray"}))
fig.update_layout(title=f"s (AR(1)) vs −media(X)/std(X)  ·  Pearson {r_p:.3f}, Spearman {r_s:.3f}",
                  xaxis_title="s = −m / σ_eq", yaxis_title="−media(X) / std(X)",
                  template="plotly_white", height=520, width=640)
fig.show()

pd.Series({"n finestre": len(cmp), "Pearson": r_p, "Spearman": r_s, "pendenza": slope,
           "intercetta": intercept, "errore assoluto medio": mae,
           "stessa zona di trading": zone_agreement}).to_frame("valore")

,valore
n finestre,691.000000
Pearson,0.994459
Spearman,0.996064
pendenza,1.003007
intercetta,0.000510
errore assoluto medio,0.066484
stessa zona di trading,0.926194


In [9]:
if r_p >= 0.95 and zone_agreement >= 0.85:
    verdict = "**regge**"
elif r_p >= 0.80:
    verdict = "**regge solo in parte**"
else:
    verdict = "**non regge**"

display(Markdown(
    f"### Conclusione\n\n"
    f"Su {len(cmp)} finestre valide l'approssimazione $s\\approx-\\overline X/\\mathrm{{std}}(X)$ {verdict}: "
    f"correlazione di Pearson {r_p:.3f} (Spearman {r_s:.3f}), pendenza della retta {slope:.2f}, "
    f"errore assoluto medio {mae:.2f}, stessa zona di trading nel {100 * zone_agreement:.0f}% dei giorni.\n\n"
    f"La stima via AR(1) resta quella da usare per il motore: l'approssimazione campionaria ignora la dinamica "
    f"(non usa b), quindi non distingue una serie che rientra velocemente da una con un trend lento che ha "
    f"la stessa media e dispersione."
))

### Conclusione

Su 691 finestre valide l'approssimazione $s\approx-\overline X/\mathrm{std}(X)$ **regge**: correlazione di Pearson 0.994 (Spearman 0.996), pendenza della retta 1.00, errore assoluto medio 0.07, stessa zona di trading nel 93% dei giorni.

La stima via AR(1) resta quella da usare per il motore: l'approssimazione campionaria ignora la dinamica (non usa b), quindi non distingue una serie che rientra velocemente da una con un trend lento che ha la stessa media e dispersione.

## 6 · Distribuzione di κ stimato e file di riferimento per il C++

Si simula un OU con parametri **noti** (passo esatto):
$X_{t+1}=m(1-e^{-\kappa\Delta t})+e^{-\kappa\Delta t}X_t+\sigma_{eq}\sqrt{1-e^{-2\kappa\Delta t}}\,Z_t$, $\Delta t=1/252$,
60 punti, e lo si stima 1000 volte con lo stesso stimatore del punto 3. Tre valori di κ: $8.4$ (esattamente sul filtro),
$30$ e $100$. La stima di $b$ sulle finestre corte è distorta verso il basso, quindi $\hat\kappa$ viene distorto verso l'alto.

In [10]:
SEED = 20261124
N_SIMS, N_POINTS = 1000, WINDOW
KAPPAS_TRUE = (8.4, 30.0, 100.0)
M_TRUE, SIGMA_EQ_TRUE = 0.0, 0.03


def simulate_ou(kappa: float, m: float, sigma_eq: float, n_points: int,
                rng: np.random.Generator) -> np.ndarray:
    '''Simulate an OU process with the exact discretisation (dt = 1/252).

    Parameters
    ----------
    kappa : float
        Mean-reversion speed (annualised, > 0).
    m : float
        Long-run mean.
    sigma_eq : float
        Stationary standard deviation (> 0).
    n_points : int
        Number of points to generate.
    rng : np.random.Generator
        Random generator (the only source of randomness).

    Returns
    -------
    np.ndarray
        Path of length ``n_points``, started from the stationary distribution.

    Raises
    ------
    ValueError
        If ``kappa`` or ``sigma_eq`` is not positive or ``n_points < 2``.

    Notes
    -----
    Exact AR(1) equivalent: ``b = exp(-kappa/252)``, ``a = m (1 - b)``,
    ``sd_zeta = sigma_eq sqrt(1 - b^2)``. Starting from the stationary law avoids burn-in.
    '''
    if kappa <= 0 or sigma_eq <= 0 or n_points < 2:
        raise ValueError("kappa, sigma_eq > 0 e n_points >= 2 richiesti")
    b = np.exp(-kappa / TRADING_DAYS)
    a = m * (1.0 - b)
    sd_zeta = sigma_eq * np.sqrt(1.0 - b * b)
    x = np.empty(n_points)
    x[0] = rng.normal(m, sigma_eq)
    shocks = rng.normal(0.0, sd_zeta, n_points - 1)
    for t in range(n_points - 1):
        x[t + 1] = a + b * x[t] + shocks[t]
    return x


# Sanity: su un campione lunghissimo lo stimatore ritrova i parametri veri (consistenza)
long_fit = estimate_ou_from_x(simulate_ou(30.0, M_TRUE, SIGMA_EQ_TRUE, 400_000, np.random.default_rng(1)))
assert abs(long_fit.kappa / 30.0 - 1) < 0.05 and abs(long_fit.sigma_eq / SIGMA_EQ_TRUE - 1) < 0.05
assert abs(long_fit.b - np.exp(-30 / TRADING_DAYS)) < 2e-3

rng = np.random.default_rng(SEED)
sim_tables: dict[float, pd.DataFrame] = {}
sim_paths: dict[float, list[np.ndarray]] = {}
for kappa_true in KAPPAS_TRUE:
    recs, paths = [], []
    for k in range(N_SIMS):
        path = simulate_ou(kappa_true, M_TRUE, SIGMA_EQ_TRUE, N_POINTS, rng)
        recs.append(vars(estimate_ou_from_x(path)))
        if k < 3:
            paths.append(path)
    sim_tables[kappa_true] = pd.DataFrame(recs)
    sim_paths[kappa_true] = paths

summary_rows = []
for kappa_true, df in sim_tables.items():
    kh = df["kappa"].dropna()
    summary_rows.append({
        "kappa vero": kappa_true, "b vero": np.exp(-kappa_true / TRADING_DAYS),
        "b medio stimato": df["b"].mean(),
        "kappa stimato: media": kh.mean(), "mediana": kh.median(), "std": kh.std(),
        "q05": kh.quantile(0.05), "q95": kh.quantile(0.95),
        "% kappa non definito": 100 * df["kappa"].isna().mean(),
        "% passa filtro": 100 * df["valid"].mean(),
    })
summary = pd.DataFrame(summary_rows).set_index("kappa vero")

# b stimato distorto verso il basso (distorsione nota dell'AR(1) su campioni corti)
assert (summary["b medio stimato"] < summary["b vero"]).all()
assert all(len(t) == N_SIMS for t in sim_tables.values())
summary.round(3)

,b vero,b medio stimato,kappa stimato: media,mediana,std,q05,q95,% kappa non definito,% passa filtro
kappa vero,,,,,,,,,
8.4,0.967,0.885,32.204,26.295,23.757,6.416,76.521,1.1,90.3
30.0,0.888,0.825,49.973,44.311,26.771,18.726,102.825,0.1,99.6
100.0,0.672,0.621,124.305,113.548,48.725,65.525,214.719,0.0,100.0


In [11]:
fig = make_subplots(rows=1, cols=len(KAPPAS_TRUE), subplot_titles=[f"κ vero = {k:g}" for k in KAPPAS_TRUE])
for j, kappa_true in enumerate(KAPPAS_TRUE, start=1):
    kh = sim_tables[kappa_true]["kappa"].dropna()
    hi = float(kh.quantile(0.99))
    fig.add_trace(go.Histogram(x=kh[kh <= hi], nbinsx=50, showlegend=False,
                               marker={"color": "steelblue"}), row=1, col=j)
    fig.add_vline(x=kappa_true, line={"color": "crimson", "dash": "dash"}, row=1, col=j)
fig.update_layout(title="Distribuzione di κ stimato (1000 simulazioni, 60 punti; rosso = κ vero; troncato al 99° percentile)",
                  template="plotly_white", height=380)
fig.update_xaxes(title_text="κ stimato")
fig.show()

### File di riferimento per i test C++

`data/ou_reference.json` contiene tre blocchi, tutti indipendenti dai dati di mercato:

* `regression_cases` — rendimenti titolo/ETF (60 punti) con atteso: α, β, residui, parametri OU e s-score;
* `estimator_cases` — serie X simulate (60 punti) con atteso: a, b, κ, m, σ_eq e $(X_{n}-m)/\sigma_{eq}$;
* `distribution_cases` — statistiche di $\hat\kappa$ su 1000 simulazioni per ciascun κ vero.

Il C++ non può riprodurre i numeri casuali di NumPy: per `distribution_cases` va confrontata la **statistica**
(entro l'errore Monte Carlo riportato), mentre per gli altri due blocchi il confronto è **esatto** (tolleranza 1e-9).
`null` = valore non definito (`NaN`).

In [12]:
def clean(v: object) -> object:
    '''Convert numpy scalars/arrays to JSON-safe Python objects (NaN -> None).

    Parameters
    ----------
    v : object
        Value to convert (float, int, bool, np.ndarray, list, dict, None).

    Returns
    -------
    object
        JSON-serialisable equivalent.
    '''
    if isinstance(v, dict):
        return {k: clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, np.ndarray)):
        return [clean(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return None if np.isnan(v) else float(v)
    return v


def ou_expected(x: np.ndarray) -> dict[str, object]:
    '''Expected estimator outputs for a series ``X`` (general s = (X_last - m)/sigma_eq).'''
    p = estimate_ou_from_x(x)
    s_gen = (x[-1] - p.m) / p.sigma_eq if p.valid else float("nan")
    return {"a": p.a, "b": p.b, "kappa": p.kappa, "m": p.m, "sigma_eq": p.sigma_eq,
            "valid": p.valid, "s_score": s_gen}


# ---- regression_cases: serie sintetiche con seme fisso -------------------------------
def make_regression_case(name: str, seed: int, mean_reverting: bool) -> dict[str, object]:
    rng_c = np.random.default_rng(seed)
    etf_ret = rng_c.normal(0.0004, 0.012, WINDOW)
    if mean_reverting:
        eps = np.diff(simulate_ou(30.0, 0.0, 0.03, WINDOW + 1, rng_c))  # residui = incrementi di un OU
    else:
        eps = rng_c.normal(0.0, 0.008, WINDOW)
    stock_ret = 0.0002 + 0.9 * etf_ret + eps
    fit = fit_ols(stock_ret, etf_ret)
    exp_ou = ou_expected(np.cumsum(fit.residuals))
    exp_ou["s_score"] = -exp_ou["m"] / exp_ou["sigma_eq"] if exp_ou["valid"] else float("nan")  # X_60 = 0
    return {"name": name, "seed": seed, "etf_returns": etf_ret, "stock_returns": stock_ret,
            "expected": {"alpha": fit.alpha, "beta": fit.beta, "residuals": fit.residuals, "ou": exp_ou}}


regression_cases = [make_regression_case("mean_reverting", 11, True),
                    make_regression_case("random_noise", 12, False)]

# ---- estimator_cases / distribution_cases ----------------------------------------------
estimator_cases = [
    {"kappa_true": kt, "m_true": M_TRUE, "sigma_eq_true": SIGMA_EQ_TRUE, "x": path, "expected": ou_expected(path)}
    for kt in KAPPAS_TRUE for path in sim_paths[kt]
]

distribution_cases = []
for kt in KAPPAS_TRUE:
    df = sim_tables[kt]
    kh = df["kappa"].dropna()
    distribution_cases.append({
        "kappa_true": kt, "b_true": float(np.exp(-kt / TRADING_DAYS)),
        "m_true": M_TRUE, "sigma_eq_true": SIGMA_EQ_TRUE,
        "n_sims": N_SIMS, "n_points": N_POINTS,
        "kappa_hat": {"n_defined": int(len(kh)), "mean": kh.mean(), "std": kh.std(),
                      "median": kh.median(), "q05": kh.quantile(0.05), "q25": kh.quantile(0.25),
                      "q75": kh.quantile(0.75), "q95": kh.quantile(0.95),
                      "mc_standard_error_of_mean": kh.std() / np.sqrt(len(kh))},
        "b_hat": {"mean": df["b"].mean(), "std": df["b"].std()},
        "frac_kappa_undefined": float(df["kappa"].isna().mean()),
        "frac_pass_filter": float(df["valid"].mean()),
    })

reference = clean({
    "meta": {
        "generator": "numpy.random.default_rng (PCG64)", "seed": SEED, "created_for": "C++ engine tests",
        "trading_days": TRADING_DAYS, "dt": 1.0 / TRADING_DAYS, "window": WINDOW, "n_pairs_ar1": WINDOW - 1,
        "b_max": B_MAX, "ols": "OLS with intercept; residuals sum to zero",
        "ar1": "X[n+1] = a + b X[n] + zeta; OLS with intercept on X[:-1], X[1:]",
        "sigma_zeta_ddof": 2,
        "formulas": {"kappa": "-252 ln b", "m": "a / (1 - b)", "sigma_eq": "sigma_zeta / sqrt(1 - b^2)",
                     "s_score": "(X_last - m) / sigma_eq  (= -m / sigma_eq when X_last = 0)"},
        "undefined_rule": "kappa, m, sigma_eq, s_score are null unless 0 < b < 1; s_score also null if b >= b_max",
        "tolerances": {"exact_cases": 1e-9, "distribution": "|mean_cpp - mean| <= 4 * mc_standard_error_of_mean"},
        "kappa_true_values": list(KAPPAS_TRUE),
    },
    "regression_cases": regression_cases,
    "estimator_cases": estimator_cases,
    "distribution_cases": distribution_cases,
})
with OU_REFERENCE_JSON.open("w", encoding="utf-8") as fh:
    json.dump(reference, fh, indent=1, allow_nan=False)
logger.info("Scritto %s (%.0f KB)", OU_REFERENCE_JSON, OU_REFERENCE_JSON.stat().st_size / 1024)

INFO xom_xle: Scritto data/ou_reference.json (30 KB)


**Test del punto 6** — rilettura del JSON e ricalcolo indipendente di tutti i valori attesi:

In [13]:
ref = json.loads(OU_REFERENCE_JSON.read_text(encoding="utf-8"))
assert {"meta", "regression_cases", "estimator_cases", "distribution_cases"} <= ref.keys()
assert len(ref["estimator_cases"]) == 3 * len(KAPPAS_TRUE) and len(ref["distribution_cases"]) == len(KAPPAS_TRUE)


def close(a: float | None, b: float | None) -> bool:
    if a is None or b is None:
        return a is None and b is None
    return bool(np.isclose(a, b, rtol=1e-9, atol=1e-12))


for case in ref["estimator_cases"]:
    again = ou_expected(np.asarray(case["x"]))
    assert len(case["x"]) == N_POINTS
    assert all(close(clean(again[k]), case["expected"][k]) for k in ("a", "b", "kappa", "m", "sigma_eq", "s_score"))
    assert again["valid"] == case["expected"]["valid"]

for case in ref["regression_cases"]:
    fit = fit_ols(np.asarray(case["stock_returns"]), np.asarray(case["etf_returns"]))
    assert len(case["stock_returns"]) == WINDOW
    assert close(fit.beta, case["expected"]["beta"]) and close(fit.alpha, case["expected"]["alpha"])
    assert np.allclose(fit.residuals, case["expected"]["residuals"], rtol=1e-9, atol=1e-12)
    assert abs(sum(case["expected"]["residuals"])) < 1e-12
    exp = case["expected"]["ou"]
    again = estimate_ou(fit.residuals)
    assert close(clean(again.b), exp["b"]) and again.valid == exp["valid"]
    if exp["valid"]:  # X_60 = 0 -> s = -m/sigma_eq
        assert close(clean(-again.m / again.sigma_eq), exp["s_score"])
    else:
        assert exp["s_score"] is None

for case in ref["distribution_cases"]:
    kh = case["kappa_hat"]
    assert kh["q05"] <= kh["q25"] <= kh["median"] <= kh["q75"] <= kh["q95"]
    assert 0.0 <= case["frac_pass_filter"] <= 1.0 and kh["n_defined"] <= N_SIMS
    assert case["b_hat"]["mean"] < case["b_true"]            # b distorto verso il basso
    assert kh["median"] > case["kappa_true"] * 0.9           # kappa non sottostimato in mediana

# deterministico: rigenerare con lo stesso seme riproduce i primi percorsi byte per byte
rng_chk = np.random.default_rng(SEED)
assert np.array_equal(simulate_ou(KAPPAS_TRUE[0], M_TRUE, SIGMA_EQ_TRUE, N_POINTS, rng_chk), sim_paths[KAPPAS_TRUE[0]][0])

logger.info("Punto 6: tutti i test superati")
logger.info("Notebook completato dall'inizio alla fine.")

INFO xom_xle: Punto 6: tutti i test superati
INFO xom_xle: Notebook completato dall'inizio alla fine.
